# Support Integrity Auditor (SIA)
**Problem Statement 01 — AI/ML Track**

Full reproducible pipeline: pseudo-labeling → classifier training → dossier generation


In [ ]:
# Install dependencies
!pip install -r requirements.txt -q

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '.')
os.makedirs('outputs', exist_ok=True)
print('Setup complete.')

## 1. Data Loading

In [ ]:
from src.data_loader import load_dataset

# Option A: Load from Kaggle dataset
# !kaggle datasets download -d ajverse/customer-support-tickets-crm-dataset -p data/ --unzip
# df = load_dataset('data/customer_support_tickets.csv')

# Option B: Synthetic data (demo)
df = load_dataset()
print(df.shape)
df.head(3)

In [ ]:
print('Priority distribution:')
print(df['Ticket Priority'].value_counts())
print('\nChannel distribution:')
print(df['Ticket Channel'].value_counts())

## 2. Stage 1 — Pseudo-Label Generation (Self-Supervised)

In [ ]:
from src.pseudo_labeler import generate_pseudo_labels, signal_agreement

df = generate_pseudo_labels(df)
df.to_csv('outputs/pseudo_labeled.csv', index=False)

print('\nMismatch type distribution:')
print(df['mismatch_type'].value_counts())
print(f'\nSignal Agreement (NLP vs RT): {signal_agreement(df):.4f}')

In [ ]:
# Visualise score distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes, ['nlp_score', 'rt_score', 'fused_score'],
                           ['NLP Score', 'RT Score', 'Fused Score']):
    for mtype, color in [('Hidden Crisis', 'red'), ('False Alarm', 'orange'), ('Consistent', 'green')]:
        subset = df[df['mismatch_type'] == mtype][col]
        if len(subset):
            ax.hist(subset, bins=20, alpha=0.5, color=color, label=mtype)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.axvline(0.5, color='black', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('outputs/score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ablation Study

In [ ]:
from src.evaluation import ablation_report
abl = ablation_report(df)
abl

## 4. Stage 2 — Classifier Training (DeBERTa-v3-small)

In [ ]:
# NOTE: Fine-tuning requires GPU for reasonable speed.
# On Colab T4: ~15-20 min for 2000 samples / 4 epochs.
# Set SKIP_TRAINING=True to use pseudo-labels only.

SKIP_TRAINING = False  # Set True if no GPU

if not SKIP_TRAINING:
    from src.classifier import train_model, predict_batch
    model, tokenizer, scaler, le_type, history = train_model(
        df, save_dir='outputs/model'
    )
    df_pred = predict_batch(df, model, tokenizer, scaler, le_type)
    df_pred.to_csv('outputs/predictions.csv', index=False)
else:
    print('Skipping training. Using pseudo-labels as predictions.')
    df_pred = df.copy()
    df_pred['pred_mismatch'] = df_pred['mismatch_label']
    df_pred['mismatch_confidence'] = df_pred['fused_score']

## 5. Evaluation Metrics

In [ ]:
from src.evaluation import evaluate

y_true = df_pred['mismatch_label'].values
y_pred = df_pred['pred_mismatch'].values
sa = signal_agreement(df_pred)

metrics = evaluate(y_true, y_pred, signal_agreement=sa)
with open('outputs/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

## 6. Stage 3 — Evidence Dossier Generation

In [ ]:
from src.dossier import generate_all_dossiers

dossiers = generate_all_dossiers(df_pred, output_path='outputs/dossiers.json')
print(f'Generated {len(dossiers)} dossiers')
print('\nSample dossier:')
print(json.dumps(dossiers[0], indent=2))

## 7. Severity Delta Heatmap

In [ ]:
import seaborn as sns

if 'Ticket Type' in df_pred.columns and 'Ticket Channel' in df_pred.columns:
    heatmap_data = df_pred.groupby(['Ticket Type', 'Ticket Channel'])['severity_delta'].mean().unstack()
    plt.figure(figsize=(10, 5))
    sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                linewidths=0.5)
    plt.title('Avg Severity Delta: Ticket Type × Channel\n(+ve = under-labeled, -ve = over-labeled)')
    plt.tight_layout()
    plt.savefig('outputs/severity_delta_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## ✅ Pipeline Complete
All outputs saved to `outputs/`:
- `pseudo_labeled.csv` — dataset with pseudo-labels
- `predictions.csv` — classifier predictions
- `dossiers.json` — evidence dossiers for mismatch tickets
- `metrics.json` — evaluation metrics
- `score_distributions.png` — signal visualisation
- `severity_delta_heatmap.png` — heatmap
